# Week 1 -- Business Understanding
## Prediksi State of Charge (SOC) Baterai Kendaraan Listrik

**Mata Kuliah:** Proyek Data Mining  
**Dataset:** LG 18650HG2 Li-Ion Battery Data -- McMaster University  

---

## 1. Case Study

### 1.1 Latar Belakang

Kendaraan listrik (Electric Vehicle/EV) semakin banyak digunakan sebagai solusi transportasi ramah lingkungan. Komponen paling kritis pada kendaraan listrik adalah **baterai Lithium-Ion**, yang menjadi satu-satunya sumber energi penggerak. Salah satu parameter terpenting dalam manajemen baterai kendaraan listrik adalah **State of Charge (SOC)** -- yaitu persentase energi yang tersisa di dalam baterai relatif terhadap kapasitas penuhnya.

SOC pada kendaraan listrik berperan analog dengan indikator bahan bakar pada kendaraan konvensional. Estimasi SOC yang akurat dibutuhkan untuk:
- Menampilkan estimasi jarak tempuh tersisa (*range estimation*) kepada pengemudi
- Mencegah *overcharge* dan *deep discharge* yang merusak baterai
- Mengoptimalkan strategi pengereman regeneratif
- Memperpanjang umur pakai baterai

SOC tidak dapat diukur secara langsung oleh sensor. Metode konvensional seperti *Coulomb Counting* rentan terhadap akumulasi error, sedangkan metode berbasis tegangan (*Open Circuit Voltage*) tidak akurat saat kendaraan sedang beroperasi (kondisi dinamis). Hal ini mendorong kebutuhan akan pendekatan berbasis **data-driven / machine learning** untuk estimasi SOC yang lebih akurat dan robust.

### 1.2 Deskripsi Dataset

Dataset yang digunakan berasal dari eksperimen yang dilakukan oleh **Dr. Phillip Kollmeyer di McMaster University, Kanada**, terhadap sel baterai **LG 18650HG2** (3Ah, 3.7V nominal). Profil daya pada drive cycle dihitung untuk satu sel baterai pada **kendaraan listrik kompak**.

| Aspek | Detail |
|---|---|
| **Sel Baterai** | LG HG2 18650, Li-ion, 3Ah, 3.7V |
| **Aplikasi** | Kendaraan listrik kompak (compact EV) |
| **Kondisi Suhu** | 6 level: -20C, -10C, 0C, 10C, 25C, 40C |
| **Drive Cycle** | UDDS (urban), HWFET (highway), LA92, US06 (agresif) -- standar pengujian otomotif |
| **Variabel Terukur** | Voltage (V), Current (A), Temperature (C), Capacity (Ah), Wh, Timestamp |
| **Sampling Rate** | 0.1 detik (drive cycles), variabel pada tes lain |
| **Format** | CSV dan MATLAB (.mat) |

Drive cycle yang digunakan merupakan standar industri otomotif:
- **UDDS** (Urban Dynamometer Driving Schedule) -- simulasi berkendara di perkotaan
- **HWFET** (Highway Fuel Economy Test) -- simulasi berkendara di jalan tol
- **LA92** -- siklus mengemudi Los Angeles, lebih agresif dari UDDS
- **US06** -- siklus agresif dengan akselerasi dan kecepatan tinggi

## 2. Define the Problem (Menentukan Masalah)

### 2.1 Problem Statement

> **Bagaimana memprediksi State of Charge (SOC) baterai kendaraan listrik secara akurat menggunakan data operasional (tegangan, arus, suhu) yang tersedia secara real-time melalui pendekatan data mining?**

### 2.2 Tipe Problem

| Aspek | Keterangan |
|---|---|
| **Jenis Task** | **Regresi** -- memprediksi nilai kontinu SOC (0%-100%) |
| **Pendekatan** | Supervised Learning |
| **Input (Features)** | Voltage, Current, Temperature, dan turunan fitur (dV/dt, dI/dt) |
| **Output (Target)** | SOC (%) -- dihitung melalui Coulomb Counting dari kolom Capacity (Ah) |

### 2.3 Tantangan Utama

1. **SOC tidak tersedia secara eksplisit** dalam dataset -- harus dihitung dari Coulomb Counting
2. **Pengaruh suhu** yang signifikan terhadap perilaku baterai kendaraan listrik, terutama di iklim ekstrem
3. **Dinamika arus** yang bervariasi sesuai pola berkendara (urban, highway, agresif)
4. **Variabel sampling rate** antar tipe pengujian
5. **Non-linearitas** hubungan antara variabel input dan SOC

## 3. Define Success (Menentukan Keberhasilan)

### 3.1 Metrik Evaluasi

| Metrik | Deskripsi | Target |
|---|---|---|
| **MAE** (Mean Absolute Error) | Rata-rata error absolut prediksi SOC | <= 2% |
| **RMSE** (Root Mean Squared Error) | Akar rata-rata kuadrat error | <= 3% |
| **R2 Score** | Koefisien determinasi | >= 0.98 |
| **Max Error** | Error maksimum pada satu titik prediksi | <= 5% |

### 3.2 Kriteria Keberhasilan

1. **Teknis:** Model mampu memprediksi SOC dengan MAE <= 2% pada data test (drive cycle yang tidak digunakan saat training)
2. **Generalisasi:** Model tetap akurat pada kondisi suhu yang berbeda dari data training, penting untuk kendaraan listrik yang beroperasi di berbagai iklim
3. **Praktis:** Model cukup ringan untuk diimplementasikan pada Battery Management System (BMS) kendaraan listrik

### 3.3 Baseline Comparison

| Metode | Estimasi Akurasi | Catatan |
|---|---|---|
| Coulomb Counting (konvensional) | MAE ~5-10% | Akumulasi error seiring waktu |
| OCV-based | MAE ~3-5% | Hanya akurat saat kendaraan berhenti (rest) |
| **Target Model ML** | **MAE <= 2%** | **Akurat saat kendaraan beroperasi (dinamis)** |

## 4. Identify Factors / Levers (Mengidentifikasi Faktor yang Mempengaruhi)

### 4.1 Faktor Langsung (Direct Features)

| Faktor | Pengaruh terhadap SOC | Kategori |
|---|---|---|
| **Voltage (V)** | Korelasi kuat -- tegangan turun seiring SOC menurun | Primary |
| **Current (A)** | Menentukan laju discharge/charge; bervariasi sesuai pola berkendara | Primary |
| **Temperature (C)** | Mempengaruhi kapasitas efektif, resistansi internal, dan karakteristik elektrokimia baterai | Primary |
| **Capacity / Ah** | Akumulasi muatan yang keluar/masuk -- basis perhitungan SOC | Derived |

### 4.2 Faktor Turunan (Engineered Features)

| Faktor | Deskripsi | Relevansi |
|---|---|---|
| **dV/dt** | Laju perubahan tegangan | Menangkap dinamika transien baterai saat akselerasi/deselerasi |
| **dI/dt** | Laju perubahan arus | Menangkap perubahan beban akibat pola mengemudi |
| **Power (W)** | V x I -- daya sesaat | Representasi beban aktual kendaraan |
| **Impedansi sesaat** | V/I pada saat ada arus | Indikator kondisi internal baterai |
| **Rolling statistics** | Mean/std Voltage dan Current dalam window waktu | Menangkap tren jangka pendek |

### 4.3 Faktor Eksternal / Kondisi Operasi

| Faktor | Pengaruh |
|---|---|
| **Ambient Temperature** | 6 level suhu (-20C s.d. 40C) -- merepresentasikan operasi EV di berbagai iklim |
| **Drive Cycle Profile** | Pola discharge berbeda (UDDS=urban, US06=agresif, HWFET=highway) sesuai gaya berkendara |
| **C-rate** | Laju discharge (0.5C, 1C, 2C) mempengaruhi kapasitas terukur |
| **Aging** | Degradasi kapasitas seiring penggunaan (minimal pada dataset ini karena sel baru) |

### 4.4 Diagram Faktor

```
                    +------------------+
                    |   SOC (Target)   |
                    +--------+---------+
           +-----------------+-----------------+
           v                 v                  v
   +--------------+  +--------------+  +----------------+
   |   Elektrik   |  |    Termal    |  |   Operasi EV   |
   |  - Voltage   |  | - Temp Cell  |  | - Drive Cycle  |
   |  - Current   |  | - Temp Amb   |  | - Gaya Kendara |
   |  - Power     |  |              |  | - C-rate       |
   |  - dV/dt     |  |              |  | - Aging        |
   |  - dI/dt     |  |              |  |                |
   +--------------+  +--------------+  +----------------+
```

## 5. Ringkasan dan Rencana Selanjutnya

| Minggu | Fase CRISP-DM | Deliverable |
|---|---|---|
| **Week 1** | Business Understanding | Dokumen ini |
| **Week 2** | Data Understanding | Eksplorasi data, kualitas data, visualisasi |
| **Week 3** | Data Preparation | Cleaning, feature engineering, SOC calculation |
| **Week 4** | Modeling | Training dan tuning model ML |
| **Week 5** | Evaluation | Evaluasi performa, perbandingan model |

### File Dataset yang Akan Digunakan

Dari seluruh dataset, file yang **relevan untuk prediksi SOC kendaraan listrik** adalah:

| Tipe File | Kegunaan | Prioritas |
|---|---|---|
| **Drive Cycles** (UDDS, HWFET, LA92, US06) | Profil berkendara realistis -- data utama untuk training/testing | Tinggi |
| **Mixed Cycles** (Mixed1-8) | Skenario berkendara campuran untuk generalisasi | Tinggi |
| **Discharge** (Dis_0p5C, Dis_2C, Cap_1C) | Discharge konstan untuk validasi | Sedang |
| **HPPC** | Karakterisasi impedansi | Referensi |
| **Charge** | Proses charging -- di-exclude jika fokus pada saat berkendara (discharge) | Rendah |

**Format yang digunakan: CSV** (identik dengan .mat, lebih mudah dibaca Python)